# Archived Spark implementation
Historical source from the previous laptop. Run the notebooks one directory above for the current portable pipeline. Outputs have been cleared to avoid presenting old results as current.

# 02c - Further graph and recommendation analysis

This notebook extends notebooks 01, 01b, 02, and 02b with analyses that directly inform playlist-completion modeling and evaluation. It measures long-tail exposure, within-playlist artist diversity, content-feature feasibility, and metadata-matching bias.

All degree statistics are descriptive full-graph quantities. Notebook 03 must recompute any degree-based model inputs from training edges only.

## 1. Environment and validated graph inputs

Run notebooks 01, 01b, 02, and 02b before this notebook.

In [ ]:
import os
import sys
from pathlib import Path


def find_project_root():
    required = Path("data/processed/gnn_integrated/playlist_track_edges.parquet")
    for anchor in [Path.cwd().resolve(), Path(sys.executable).resolve()]:
        for parent in [anchor, *anchor.parents]:
            for candidate in [parent, parent / "art_xharra"]:
                if (candidate / required).exists():
                    return candidate.resolve()
    raise FileNotFoundError("Integrated graph outputs are missing. Run notebook 02 first.")


PROJECT_ROOT = find_project_root()
JDK_ROOT = PROJECT_ROOT / ".tools/jdk17/jdk-17.0.20+8"
HADOOP_ROOT = PROJECT_ROOT / ".tools/hadoop"
os.environ["JAVA_HOME"] = str(JDK_ROOT)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["PATH"] = str(JDK_ROOT / "bin") + os.pathsep + os.environ.get("PATH", "")
if os.name == "nt":
    os.environ["HADOOP_HOME"] = str(HADOOP_ROOT)
    os.environ["PATH"] = str(HADOOP_ROOT / "bin") + os.pathsep + os.environ["PATH"]

from pyspark import StorageLevel
from pyspark.sql import SparkSession, functions as F
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

spark = (
    SparkSession.builder.master("local[*]")
    .appName("Integrated-Graph-Further-Analysis")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .config("spark.driver.memory", "4g")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

GRAPH_ROOT = PROJECT_ROOT / "data/processed/gnn_integrated"
FIGURE_ROOT = PROJECT_ROOT / "reports/figures/data_analysis_further"
REPORTS_ROOT = PROJECT_ROOT / "reports"
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

playlist_nodes = spark.read.parquet(str(GRAPH_ROOT / "playlist_nodes.parquet")).persist(StorageLevel.MEMORY_AND_DISK)
track_nodes = spark.read.parquet(str(GRAPH_ROOT / "track_nodes.parquet")).persist(StorageLevel.MEMORY_AND_DISK)
artist_nodes = spark.read.parquet(str(GRAPH_ROOT / "artist_nodes.parquet")).persist(StorageLevel.MEMORY_AND_DISK)
playlist_track_edges = spark.read.parquet(str(GRAPH_ROOT / "playlist_track_edges.parquet")).persist(StorageLevel.MEMORY_AND_DISK)
track_artist_edges = spark.read.parquet(str(GRAPH_ROOT / "track_artist_edges.parquet")).persist(StorageLevel.MEMORY_AND_DISK)

analysis_counts = {
    "playlists": playlist_nodes.count(),
    "tracks": track_nodes.count(),
    "artists": artist_nodes.count(),
    "playlist-track edges": playlist_track_edges.count(),
}
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 180, "axes.titleweight": "bold"})
display(pd.DataFrame(analysis_counts.items(), columns=["graph object", "count"]))
print(f"Figures: {FIGURE_ROOT.relative_to(PROJECT_ROOT)}")

## 2. Long-tail catalog exposure

A recommender can look accurate while repeatedly exposing only a small head of popular tracks. The two panels compare node share with edge share and show the Lorenz curve of playlist-edge exposure.

In [ ]:
track_degree = (
    playlist_track_edges.groupBy("dst_track_node_id").count()
    .withColumnRenamed("count", "playlist_degree")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

track_degree_bands = (
    track_degree.withColumn(
        "degree_band",
        F.when(F.col("playlist_degree") == 1, "1 playlist")
        .when(F.col("playlist_degree") <= 4, "2-4 playlists")
        .when(F.col("playlist_degree") <= 19, "5-19 playlists")
        .otherwise("20+ playlists"),
    )
    .withColumn(
        "degree_band_id",
        F.when(F.col("playlist_degree") == 1, 0)
        .when(F.col("playlist_degree") <= 4, 1)
        .when(F.col("playlist_degree") <= 19, 2)
        .otherwise(3),
    )
)
degree_band_pdf = (
    track_degree_bands.groupBy("degree_band_id", "degree_band")
    .agg(F.count("*").alias("track_count"), F.sum("playlist_degree").alias("edge_count"))
    .orderBy("degree_band_id").toPandas()
)
degree_band_pdf["catalog_share_pct"] = 100 * degree_band_pdf["track_count"] / degree_band_pdf["track_count"].sum()
degree_band_pdf["edge_share_pct"] = 100 * degree_band_pdf["edge_count"] / degree_band_pdf["edge_count"].sum()

degree_values = np.sort(track_degree.select("playlist_degree").toPandas()["playlist_degree"].to_numpy(dtype=float))
lorenz_x = np.arange(degree_values.size + 1) / degree_values.size
lorenz_y = np.concatenate(([0.0], np.cumsum(degree_values) / degree_values.sum()))
degree_gini = float(1 - 2 * np.trapezoid(lorenz_y, lorenz_x))

def top_edge_share(percent):
    head_size = max(1, int(np.ceil(degree_values.size * percent / 100)))
    return 100 * degree_values[-head_size:].sum() / degree_values.sum()

concentration_summary_pdf = pd.DataFrame({
    "metric": ["Track-degree Gini", "Edges held by top 1%", "Edges held by top 5%", "Edges held by top 10%"],
    "value": [degree_gini, top_edge_share(1), top_edge_share(5), top_edge_share(10)],
    "unit": ["0-1", "%", "%", "%"],
})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_pdf = degree_band_pdf.melt(
    id_vars=["degree_band_id", "degree_band"],
    value_vars=["catalog_share_pct", "edge_share_pct"],
    var_name="share_type", value_name="percentage",
)
plot_pdf["share_type"] = plot_pdf["share_type"].map({
    "catalog_share_pct": "Share of tracks", "edge_share_pct": "Share of playlist edges",
})
sns.barplot(data=plot_pdf, x="degree_band", y="percentage", hue="share_type", palette="colorblind", ax=axes[0])
axes[0].set_title("Catalog share versus playlist exposure")
axes[0].set_xlabel("Track playlist-degree band")
axes[0].set_ylabel("Share (%)")
axes[0].tick_params(axis="x", rotation=25)
axes[0].legend(title="")

axes[1].plot(lorenz_x, lorenz_y, color=sns.color_palette("colorblind")[1], linewidth=2, label=f"Observed (Gini={degree_gini:.3f})")
axes[1].plot([0, 1], [0, 1], "--", color="gray", label="Equal exposure")
axes[1].set_title("Lorenz curve of track exposure")
axes[1].set_xlabel("Cumulative share of tracks")
axes[1].set_ylabel("Cumulative share of playlist edges")
axes[1].legend()
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "track_exposure_concentration.png", bbox_inches="tight")
plt.show()
display(degree_band_pdf.round(3))
display(concentration_summary_pdf.round(3))

**Observed.** Exposure is substantially concentrated. Although 62.26% of tracks occur in exactly one playlist, they account for only 25.22% of edges; the 1.12% of tracks connected to at least 20 playlists receive 16.61% of edges. The track-degree Gini coefficient is 0.499, and the top 10% of tracks receive 46.09% of all playlist edges. This supports reporting long-tail coverage alongside ranking accuracy.

## 3. Artist diversity within playlists

The distinct-artist ratio is `artist_count / track_count`. A value near 1 means most tracks come from different artists; a lower value indicates more repeated artists.

In [ ]:
playlist_band_order = [
    "under_5_not_eligible", "small_5_9", "medium_10_24",
    "large_25_99", "very_large_100_plus",
]
playlist_diversity = playlist_nodes.select(
    "playlist_node_id", "playlist_size_band", "track_count", "artist_count"
).withColumn("artist_diversity_ratio", F.col("artist_count") / F.col("track_count"))

diversity_summary_pdf = (
    playlist_diversity.groupBy("playlist_size_band")
    .agg(
        F.count("*").alias("playlist_count"),
        F.mean("artist_diversity_ratio").alias("mean_diversity_ratio"),
        F.expr("percentile_approx(artist_diversity_ratio, 0.25, 10000)").alias("q1_diversity_ratio"),
        F.expr("percentile_approx(artist_diversity_ratio, 0.50, 10000)").alias("median_diversity_ratio"),
        F.expr("percentile_approx(artist_diversity_ratio, 0.75, 10000)").alias("q3_diversity_ratio"),
    ).toPandas().set_index("playlist_size_band").reindex(playlist_band_order).reset_index()
)
playlist_diversity_pdf = playlist_diversity.toPandas()
playlist_diversity_pdf["playlist_size_band"] = pd.Categorical(
    playlist_diversity_pdf["playlist_size_band"], categories=playlist_band_order, ordered=True
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.boxplot(
    data=playlist_diversity_pdf, x="playlist_size_band", y="artist_diversity_ratio",
    order=playlist_band_order, showfliers=False, color=sns.color_palette("colorblind")[2], ax=axes[0],
)
axes[0].set_title("Within-playlist artist diversity")
axes[0].set_xlabel("Playlist size segment")
axes[0].set_ylabel("Distinct artists / tracks")
axes[0].tick_params(axis="x", rotation=30)
sns.barplot(
    data=diversity_summary_pdf, x="playlist_size_band", y="median_diversity_ratio",
    color=sns.color_palette("colorblind")[0], ax=axes[1],
)
axes[1].set_title("Median artist diversity by size")
axes[1].set_xlabel("Playlist size segment")
axes[1].set_ylabel("Median distinct-artists ratio")
axes[1].tick_params(axis="x", rotation=30)
axes[1].set_ylim(0, 1)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "playlist_artist_diversity.png", bbox_inches="tight")
plt.show()
display(diversity_summary_pdf.round(3))

**Observed.** Artist repetition increases with playlist size. Median distinct-artist ratio declines from 1.000 for connected playlists under five tracks to 0.857 for 5–9 tracks, 0.778 for 10–24, 0.690 for 25–99, and 0.575 for playlists with at least 100 tracks. Playlist-size strata therefore represent meaningfully different completion contexts.

## 4. Audio-feature feasibility by playlist size

Overall audio coverage can hide large differences between playlist segments. The second panel applies the five-match threshold identified in notebook 02 as a practical minimum for a content-restricted experiment.

In [ ]:
audio_by_size_pdf = (
    playlist_nodes.groupBy("playlist_size_band")
    .agg(
        F.count("*").alias("playlist_count"),
        F.mean("audio_coverage_ratio").alias("mean_audio_coverage"),
        F.expr("percentile_approx(audio_coverage_ratio, 0.50, 10000)").alias("median_audio_coverage"),
        (100 * F.mean(F.when(F.col("audio_matched_track_count") == 0, 1.0).otherwise(0.0))).alias("zero_audio_match_pct"),
        (100 * F.mean(F.when(F.col("audio_matched_track_count") >= 5, 1.0).otherwise(0.0))).alias("five_plus_audio_matches_pct"),
    ).toPandas().set_index("playlist_size_band").reindex(playlist_band_order).reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
coverage_long_pdf = audio_by_size_pdf.melt(
    id_vars=["playlist_size_band"],
    value_vars=["mean_audio_coverage", "median_audio_coverage"],
    var_name="statistic", value_name="coverage_ratio",
)
coverage_long_pdf["statistic"] = coverage_long_pdf["statistic"].map({
    "mean_audio_coverage": "Mean", "median_audio_coverage": "Median",
})
sns.barplot(data=coverage_long_pdf, x="playlist_size_band", y="coverage_ratio", hue="statistic", palette="colorblind", ax=axes[0])
axes[0].set_title("Audio coverage by playlist size")
axes[0].set_xlabel("Playlist size segment")
axes[0].set_ylabel("Matched-track ratio")
axes[0].tick_params(axis="x", rotation=30)
axes[0].legend(title="")
sns.barplot(
    data=audio_by_size_pdf, x="playlist_size_band", y="five_plus_audio_matches_pct",
    color=sns.color_palette("colorblind")[3], ax=axes[1],
)
axes[1].set_title("Playlists with at least five audio matches")
axes[1].set_xlabel("Playlist size segment")
axes[1].set_ylabel("Playlists meeting threshold (%)")
axes[1].tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "audio_feasibility_by_playlist_size.png", bbox_inches="tight")
plt.show()
display(audio_by_size_pdf.round(4))

**Observed.** Audio coverage remains low in every size segment (mean 3.10%–5.10%). The median is zero through the 10–24 segment, and only 1.76% of medium playlists have at least five matched tracks. That threshold is reached by 16.65% of large and 57.75% of very large playlists, showing that the 3,440-playlist audio-feasible subset is strongly biased toward longer playlists.

## 5. Metadata-matching bias

Coverage is useful only if its missingness is understood. These summaries compare graph degree for matched and unmatched tracks/artists, and compare SPUD popularity for tracks with and without audio metadata.

In [ ]:
track_match_analysis = (
    track_nodes.select("track_node_id", "audio_features_available", "spud_popularity")
    .join(track_degree, track_nodes.track_node_id == track_degree.dst_track_node_id, "inner")
    .drop("dst_track_node_id")
)
track_match_bias_pdf = (
    track_match_analysis.groupBy("audio_features_available")
    .agg(
        F.count("*").alias("node_count"),
        F.mean("spud_popularity").alias("mean_spud_popularity"),
        F.expr("percentile_approx(spud_popularity, 0.50, 10000)").alias("median_spud_popularity"),
        F.mean("playlist_degree").alias("mean_graph_degree"),
        F.expr("percentile_approx(playlist_degree, 0.50, 10000)").alias("median_graph_degree"),
        F.expr("percentile_approx(playlist_degree, 0.90, 10000)").alias("p90_graph_degree"),
    ).orderBy("audio_features_available").toPandas()
)
track_match_bias_pdf["metadata_status"] = track_match_bias_pdf["audio_features_available"].map({0: "Audio missing", 1: "Audio available"})

artist_degree = track_artist_edges.groupBy("dst_artist_node_id").count().withColumnRenamed("count", "track_degree")
artist_match_analysis = (
    artist_nodes.select("artist_node_id", "artist_metadata_available")
    .join(artist_degree, artist_nodes.artist_node_id == artist_degree.dst_artist_node_id, "inner")
    .drop("dst_artist_node_id")
)
artist_match_bias_pdf = (
    artist_match_analysis.groupBy("artist_metadata_available")
    .agg(
        F.count("*").alias("node_count"),
        F.mean("track_degree").alias("mean_graph_degree"),
        F.expr("percentile_approx(track_degree, 0.50, 10000)").alias("median_graph_degree"),
        F.expr("percentile_approx(track_degree, 0.90, 10000)").alias("p90_graph_degree"),
    ).orderBy("artist_metadata_available").toPandas()
)
artist_match_bias_pdf["metadata_status"] = artist_match_bias_pdf["artist_metadata_available"].map({0: "Catalog missing", 1: "Catalog available"})

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(data=track_match_bias_pdf, x="metadata_status", y="mean_graph_degree", color=sns.color_palette("colorblind")[0], ax=axes[0])
axes[0].set_title("Track degree by audio-match status")
axes[0].set_xlabel("")
axes[0].set_ylabel("Mean connected playlists")
sns.barplot(data=artist_match_bias_pdf, x="metadata_status", y="mean_graph_degree", color=sns.color_palette("colorblind")[2], ax=axes[1])
axes[1].set_title("Artist degree by catalog-match status")
axes[1].set_xlabel("")
axes[1].set_ylabel("Mean connected tracks")
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "metadata_matching_bias.png", bbox_inches="tight")
plt.show()
display(track_match_bias_pdf.round(4))
display(artist_match_bias_pdf.round(4))

**Observed.** Metadata matching is not random. Audio-matched tracks have much higher mean SPUD popularity (0.327 versus 0.117) and mean playlist degree (4.14 versus 2.42). Catalog-matched artists average 12.45 connected tracks versus 3.40 for unmatched artists. Content-aware experiments will therefore overrepresent more popular and better-connected nodes unless this selection effect is reported.

## 6. Save tables and modeling implications

In [ ]:
degree_band_pdf.to_csv(REPORTS_ROOT / "track_degree_band_exposure.csv", index=False)
concentration_summary_pdf.to_csv(REPORTS_ROOT / "track_exposure_concentration.csv", index=False)
diversity_summary_pdf.to_csv(REPORTS_ROOT / "playlist_artist_diversity.csv", index=False)
audio_by_size_pdf.to_csv(REPORTS_ROOT / "audio_feasibility_by_playlist_size.csv", index=False)
track_match_bias_pdf.to_csv(REPORTS_ROOT / "track_metadata_match_bias.csv", index=False)
artist_match_bias_pdf.to_csv(REPORTS_ROOT / "artist_metadata_match_bias.csv", index=False)

top_1_share = float(concentration_summary_pdf.loc[concentration_summary_pdf["metric"] == "Edges held by top 1%", "value"].iloc[0])
overall_diversity_median = float(playlist_diversity.approxQuantile("artist_diversity_ratio", [0.5], 0.001)[0])
audio_eligible_count = playlist_nodes.where("audio_matched_track_count >= 5").count()
matched_track_mean_degree = float(track_match_bias_pdf.loc[track_match_bias_pdf["audio_features_available"] == 1, "mean_graph_degree"].iloc[0])
unmatched_track_mean_degree = float(track_match_bias_pdf.loc[track_match_bias_pdf["audio_features_available"] == 0, "mean_graph_degree"].iloc[0])

print("FURTHER ANALYSIS OBSERVATIONS")
print("=============================")
print(f"1. Track exposure has a Gini coefficient of {degree_gini:.3f}; the top 1% of tracks receive {top_1_share:.2f}% of playlist edges.")
print(f"2. The median playlist has a distinct-artist ratio of {overall_diversity_median:.3f}.")
print(f"3. Only {audio_eligible_count:,} of {analysis_counts['playlists']:,} playlists have at least five audio-matched tracks.")
print(f"4. Audio-matched tracks have mean playlist degree {matched_track_mean_degree:.2f}, versus {unmatched_track_mean_degree:.2f} for unmatched tracks.")
print("5. Report head/tail recommendation performance and catalog coverage alongside Recall@K and NDCG@K.")
print("6. Treat audio and artist metadata as optional side information; report matched-subset results separately from full-graph results.")
print("7. Stratify evaluation by playlist size because diversity and audio feasibility differ across segments.")
print(f"8. Saved four figures to {FIGURE_ROOT.relative_to(PROJECT_ROOT)} and six supporting CSV tables to reports/.")

**Overall implication.** Notebook 03 should use a collaborative graph model as the primary system, keep popularity/full-graph degree out of leakage-safe inputs, and report both ranking accuracy and exposure coverage. Content-aware ablations should clearly state their much smaller, non-random matched subset.